# 🟢 Solution: Implement Max Pool 2D

$$H_{out} = \lfloor (H - k) / s \rfloor + 1$$

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def max_pool2d(x, kernel_size, stride=None):
    """
    从零实现二维最大池化。

    最大池化在特征图的空间维度上滑动窗口，取每个窗口内的最大值。
    这是一种下采样操作，常用于 CNN 中减少空间分辨率。

    参数:
        x: 输入张量, shape = (B, C, H, W)
            B = batch size, C = 通道数, H = 高度, W = 宽度
        kernel_size: 池化窗口大小 (正方形)
        stride: 窗口步长，默认等于 kernel_size (不重叠)

    返回: (B, C, H_out, W_out)，其中
        H_out = (H - kernel_size) // stride + 1
        W_out = (W - kernel_size) // stride + 1
    """
    if stride is None:
        stride = kernel_size  # 默认不重叠

    # ---- Step 1: 用 unfold 提取滑动窗口 ----
    # x.unfold(dim, size, step) 在指定维度上提取大小为 size、步长为 step 的窗口
    # x: (B, C, H, W)
    # .unfold(2, k, s): 在 H 维度上提取窗口 → (B, C, H_out, W, k)
    # .unfold(3, k, s): 在 W 维度上提取窗口 → (B, C, H_out, W_out, k, k)
    # 最后两维 (k, k) 就是一个池化窗口内的所有元素
    patches = x.unfold(2, kernel_size, stride).unfold(3, kernel_size, stride)
    # patches shape: (B, C, H_out, W_out, kernel_size, kernel_size)

    # ---- Step 2: 取每个窗口的最大值 ----
    # .flatten(-2) 把最后两维 (k, k) 合并为一维 (k*k,)
    # 新 shape: (B, C, H_out, W_out, k*k)
    # .max(dim=-1).values 取最后一维的最大值
    # 结果 shape: (B, C, H_out, W_out)
    return patches.flatten(-2).max(dim=-1).values

In [ ]:
import torch.nn.functional as F

x = torch.randn(1, 1, 4, 4)
out = max_pool2d(x, kernel_size=2, stride=2)
ref = F.max_pool2d(x, kernel_size=2, stride=2)
print("Output shape:", out.shape)
print("Matches F.max_pool2d:", torch.allclose(out, ref))

In [ ]:
from torch_judge import check
check("max_pool2d")

## 📝 核心思路总结

1. **unfold 是滑动窗口利器**：`x.unfold(dim, size, step)` 在指定维度上提取固定大小的窗口，自动处理边界和步长。
2. **输出尺寸公式**：`H_out = (H - k) // stride + 1`，面试必背；stride=k 时不重叠，stride<k 时有重叠。
3. **flatten + max 取池化值**：把窗口的 2D 区域展平为 1D，再取 max，等价于在 (k, k) 区域取最大值。
4. **纯循环版是备选**：如果面试官不让用 unfold，嵌套循环 + slice + amax 也能实现，只是效率低。